# CISI Corpus Text Preprocessing Pipeline

This notebook implements a complete text preprocessing and normalization pipeline using the **CISI corpus** as the input dataset.

### Pipeline Steps Included:
1. **Dataset Loading & Parsing**: Downloading and extracting the CISI textual fields.
2. **Case Normalization**: Converting text to lowercase.
3. **Diacritics Removal**: Stripping accents and normalizing unicode characters.
4. **Jargon, Abbreviation & URL Normalization**: Cleaning non-standard textual noise and expanding short forms.
5. **Tokenization**: Segmenting sentences into individual token units.
6. **Noise & Punctuation Removal**: Eliminating numbers, punctuation signs, and non-informative symbols.
7. **Stopword Removal**: Filtering out high-frequency words with low semantic value.
8. **Stemming**: Reducing words to their crude base/root form.
9. **Lemmatization**: Resolving tokens to their dictionary form (lemma) using linguistic context.

---
## 0. Environment Setup & Dependencies
We start by installing and importing required libraries like `nltk` for NLP tasks and `unicodedata` for text normalization.

In [6]:
import importlib
import subprocess
import sys

librerias_proyecto = {"numpy": "numpy",
             "pandas": "pandas",
             "matplotlib":"matplotlib",
             "seaborn":"seaborn",
             "openpyxl": "openpyxl",
             "nltk": "nltk",
             "requests": "requests",
             "contractions": "contractions",
             }
             
print("====== INICIANDO VERIFICACIÓN DE ENTORNO ======\n")

# 2. Recorrer y validar cada librería de la lista
for nombre_importar, nombre_pip in librerias_proyecto.items():
    instalada = False
    
    while not instalada:
        try:
            # Intenta cargar la librería dinámicamente
            modulo = importlib.import_module(nombre_importar)
            instalada = True
            
            # Obtener la versión de forma segura
            version = getattr(modulo, "__version__", "Versión no expuesta")
            print(f"[{nombre_importar}] Lista para usar. Versión: {version}")
            
        except ImportError:
            print(f"[{nombre_importar}] No encontrada. Instalando vía pip: '{nombre_pip}'...")
            try:
                # Instala usando el entorno de Python que está ejecutando el script
                subprocess.check_call([sys.executable, "-m", "pip", "install", nombre_pip])
                print(f"[{nombre_importar}] Instalación enviada. Validando acceso...")
            except subprocess.CalledProcessError:
                print(f"[ERROR CRÍTICO] Falló la instalación de '{nombre_pip}'.")
                print("Revisa tu conexión a internet o los permisos de administrador.\n")
                break  # Rompe el ciclo while de esta librería para pasar a la siguiente

print("\n====== PROCESO DE CONFIGURACIÓN FINALIZADO ======")

====== INICIANDO VERIFICACIÓN DE ENTORNO ======

[numpy] Lista para usar. Versión: 2.5.1
[pandas] Lista para usar. Versión: 3.0.3
[matplotlib] Lista para usar. Versión: 3.11.1
[seaborn] Lista para usar. Versión: 0.13.2
[openpyxl] Lista para usar. Versión: 3.1.5
[nltk] Lista para usar. Versión: 3.10.3
[requests] Lista para usar. Versión: 2.33.1
[contractions] No encontrada. Instalando vía pip: 'contractions'...
[contractions] Instalación enviada. Validando acceso...
[contractions] Lista para usar. Versión: Versión no expuesta

====== PROCESO DE CONFIGURACIÓN FINALIZADO ======


## 1. Setup

Install/import the libraries used in the pipeline:

- `nltk` — tokenization, stopwords, stemming, lemmatization, POS tagging
- `contractions` — expands informal contractions ("don't" → "do not") as part of slang/jargon normalization
- `unicodedata` (standard library) — strips diacritics/accents
- `re` (standard library) — regex-based noise removal
- `pandas` — tabular handling of the corpus and results
- `matplotlib` — small before/after visual comparison

In [17]:
import re
import os
import string
import unicodedata
import collections
from pathlib import Path

import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

import contractions

# Download essential NLTK resources
NLTK_PACKAGES = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]
for pkg in NLTK_PACKAGES:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Could not download '{pkg}': {e}")

print("Setup complete.")

Setup complete.



## 3. Building the pipeline — one technique at a time

Each preprocessing technique is implemented as a small, independent, well-documented function
first, then combined into a single configurable class in Section 4. Building them independently
makes it easy to inspect what each step does to the text.

We'll use a short synthetic sentence that packs in several of the phenomena the pipeline must
handle (URL, email, accents, numbers, punctuation, an informal contraction, and a couple of
abbreviations), in addition to the real CISI documents.


In [18]:

demo_text = (
    "Café résumé: the 18th Ed. of the Naïve Bayes Classification System — see "
    "http://example.com/paper, or e-mail info@example.com. It's approx. 1876 vs. 1971; "
    "don't u think the govt. dept. should update the catalog #libraries!!"
)
print(demo_text)


Café résumé: the 18th Ed. of the Naïve Bayes Classification System — see http://example.com/paper, or e-mail info@example.com. It's approx. 1876 vs. 1971; don't u think the govt. dept. should update the catalog #libraries!!



### 3.1 Case normalization

Converting all text to lowercase so that, e.g., `"Library"`, `"library"`, and `"LIBRARY"` are
treated as the same token. This is done early, since several later steps (stopword lists,
abbreviation dictionary lookups) are easiest to match in a single case.


In [19]:

def normalize_case(text: str) -> str:
    '''Lowercase the text.'''
    return text.lower()


print(normalize_case(demo_text))


café résumé: the 18th ed. of the naïve bayes classification system — see http://example.com/paper, or e-mail info@example.com. it's approx. 1876 vs. 1971; don't u think the govt. dept. should update the catalog #libraries!!



### 3.2 Diacritics (accent) removal

Accented characters (e.g. `é`, `ï`, `ñ`) are normalized to their closest plain-ASCII form using
Unicode's NFKD decomposition, which separates a base letter from its combining accent mark, and
then discarding the combining marks. This ensures `"café"` and `"cafe"` are treated as the same
token.


In [20]:

def remove_diacritics(text: str) -> str:
    '''Strip accents/diacritics, e.g. 'café' -> 'cafe', 'naïve' -> 'naive'.'''
    nfkd_form = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in nfkd_form if not unicodedata.combining(ch))


print(remove_diacritics(normalize_case(demo_text)))


cafe resume: the 18th ed. of the naive bayes classification system — see http://example.com/paper, or e-mail info@example.com. it's approx. 1876 vs. 1971; don't u think the govt. dept. should update the catalog #libraries!!



### 3.3 Jargon, slang, and abbreviation normalization

Two complementary techniques are used here:

1. **Contraction expansion** (`don't` → `do not`, `it's` → `it is`) via the `contractions`
   library — this is standard informal-language normalization.
2. **Domain/abbreviation dictionary lookup** — a small, extensible mapping of common
   abbreviations, internet slang, and library/information-science shorthand (`govt.` →
   `government`, `dept.` → `department`, `approx.` → `approximately`, `u` → `you`, etc.) applied
   at the token level after tokenization. The dictionary is easy to extend for a specific domain
   or corpus.


In [21]:

# A small, extensible abbreviation / slang normalization dictionary.
# Keys are matched against lowercased tokens (punctuation already stripped where relevant).
ABBREVIATION_DICT = {
    "govt": "government",
    "dept": "department",
    "univ": "university",
    "info": "information",
    "libs": "libraries",
    "lib": "library",
    "approx": "approximately",
    "eg": "for example",
    "ie": "that is",
    "etc": "etcetera",
    "vs": "versus",
    "u": "you",
    "ur": "your",
    "thx": "thanks",
    "pls": "please",
    "w": "with",
    "wo": "without",
}


def expand_contractions(text: str) -> str:
    '''Expand informal contractions, e.g. don't -> do not, it's -> it is.'''
    return contractions.fix(text)


def normalize_abbreviations(tokens, abbrev_dict=ABBREVIATION_DICT):
    '''Replace known abbreviations/slang tokens with their expanded form.'''
    return [abbrev_dict.get(tok, tok) for tok in tokens]


demo_expanded = expand_contractions(demo_text)
print("After contraction expansion:\n", demo_expanded)


After contraction expansion:
 Café résumé: the 18th Ed. of the Naïve Bayes Classification System — see http://example.com/paper, or e-mail info@example.com. It is approx. 1876 vs. 1971; do not you think the govt. dept. should update the catalog #libraries!!



### 3.4 Noise removal

Strips content that carries little to no topical/informational value for downstream text
analysis or retrieval: URLs, email addresses, HTML/markup fragments, punctuation, standalone
numbers, and leftover extra whitespace. Very short leftover tokens (length ≤ 1) are also
dropped, as they are almost always artifacts of punctuation stripping rather than real words.


In [22]:

URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
EMAIL_RE = re.compile(r"\S+@\S+")
HTML_TAG_RE = re.compile(r"<.*?>")
MULTISPACE_RE = re.compile(r"\s+")
PURE_NUMBER_RE = re.compile(r"^\d+([.,:/]\d+)*$")


def remove_noise_text(text: str) -> str:
    '''Remove URLs, emails, and HTML tags at the raw-text level (before tokenization).'''
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = HTML_TAG_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text


def strip_punctuation(token: str) -> str:
    '''Remove punctuation characters from within a token (e.g. 'vs.' -> 'vs', 'e-mail' -> 'email').'''
    return token.translate(str.maketrans("", "", string.punctuation))


def remove_noise_tokens(tokens):
    '''
    Token-level noise removal:
      - strip punctuation from each token
      - drop tokens that become empty
      - drop pure numbers (e.g. '1876')
      - drop leftover single-character tokens
    '''
    cleaned = []
    for tok in tokens:
        tok = strip_punctuation(tok)
        if not tok:
            continue
        if PURE_NUMBER_RE.match(tok):
            continue
        if len(tok) <= 1:
            continue
        cleaned.append(tok)
    return cleaned


print("Text-level noise removal:\n", remove_noise_text(demo_expanded))


Text-level noise removal:
 Café résumé: the 18th Ed. of the Naïve Bayes Classification System — see or e-mail It is approx. 1876 vs. 1971; do not you think the govt. dept. should update the catalog #libraries!!



### 3.5 Tokenization

Splitting the cleaned text into individual word tokens, using NLTK's `word_tokenize`
(a Penn-Treebank-style tokenizer that correctly separates punctuation from words and handles
common English tokenization edge cases).


In [23]:

def tokenize(text: str):
    '''Split text into word tokens.'''
    return word_tokenize(text)


demo_clean_text = remove_diacritics(normalize_case(remove_noise_text(expand_contractions(demo_text))))
demo_tokens = tokenize(demo_clean_text)
print(demo_tokens)


['cafe', 'resume', ':', 'the', '18th', 'ed', '.', 'of', 'the', 'naive', 'bayes', 'classification', 'system', '—', 'see', 'or', 'e-mail', 'it', 'is', 'approx', '.', '1876', 'vs.', '1971', ';', 'do', 'not', 'you', 'think', 'the', 'govt', '.', 'dept', '.', 'should', 'update', 'the', 'catalog', '#', 'libraries', '!', '!']



### 3.6 Stopword removal

Removing high-frequency, low-information function words (`the`, `of`, `and`, ...) using NLTK's
built-in English stopword list. This is applied **after** noise removal and tokenization so
that punctuation attached to a stopword doesn't prevent a match.


In [24]:

STOPWORDS = set(stopwords.words("english"))
print(f"{len(STOPWORDS)} English stopwords loaded, e.g.: {sorted(list(STOPWORDS))[:15]}")


def remove_stopwords(tokens, stopword_set=STOPWORDS):
    '''Filter out stopwords (case-insensitive).'''
    return [tok for tok in tokens if tok.lower() not in stopword_set]


demo_tokens_denoised = remove_noise_tokens(demo_tokens)
demo_tokens_normalized = normalize_abbreviations(demo_tokens_denoised)
demo_tokens_nostop = remove_stopwords(demo_tokens_normalized)
print("Tokens after noise removal + abbreviation normalization:\n", demo_tokens_normalized)
print("\nTokens after stopword removal:\n", demo_tokens_nostop)


198 English stopwords loaded, e.g.: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]
Tokens after noise removal + abbreviation normalization:
 ['cafe', 'resume', 'the', '18th', 'ed', 'of', 'the', 'naive', 'bayes', 'classification', 'system', 'see', 'or', 'email', 'it', 'is', 'approximately', 'versus', 'do', 'not', 'you', 'think', 'the', 'government', 'department', 'should', 'update', 'the', 'catalog', 'libraries']

Tokens after stopword removal:
 ['cafe', 'resume', '18th', 'ed', 'naive', 'bayes', 'classification', 'system', 'see', 'email', 'approximately', 'versus', 'think', 'government', 'department', 'update', 'catalog', 'libraries']



### 3.7 Stemming

Reducing each word to a crude root form by chopping off common suffixes, using the
**Porter Stemmer** — the classic, fast stemming algorithm for English (used in the original
IR literature this corpus comes from). Stemming is aggressive and can produce non-dictionary
forms (e.g. `"classification"` → `"classif"`), but it is very effective at conflating word
variants for retrieval/indexing tasks.


In [25]:

porter = PorterStemmer()


def stem_tokens(tokens):
    '''Apply Porter stemming to each token.'''
    return [porter.stem(tok) for tok in tokens]


demo_stems = stem_tokens(demo_tokens_nostop)
print(demo_stems)


['cafe', 'resum', '18th', 'ed', 'naiv', 'bay', 'classif', 'system', 'see', 'email', 'approxim', 'versu', 'think', 'govern', 'depart', 'updat', 'catalog', 'librari']



### 3.8 Lemmatization

Reducing each word to its dictionary base form (lemma) using WordNet's `WordNetLemmatizer`.
Unlike stemming, lemmatization uses vocabulary and morphological analysis to return real words
(e.g. `"classification"` stays `"classification"`, but `"editions"` → `"edition"`,
`"published"` → `"publish"`). Accuracy is greatly improved by first tagging each token with its
part of speech (noun/verb/adjective/adverb) using NLTK's POS tagger and passing that into the
lemmatizer, since the same surface form can lemmatize differently depending on its role
(e.g. `"studies"` as a noun → `"study"`, as a verb → `"study"`, but `"better"` as an adjective
stays `"better"` while as an adverb → `"well"`).


In [26]:

lemmatizer = WordNetLemmatizer()


def _penn_to_wordnet_pos(penn_tag: str):
    '''Map a Penn Treebank POS tag to the simplified tag WordNetLemmatizer expects.'''
    if penn_tag.startswith("J"):
        return wordnet.ADJ
    if penn_tag.startswith("V"):
        return wordnet.VERB
    if penn_tag.startswith("N"):
        return wordnet.NOUN
    if penn_tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN  # default fallback


def lemmatize_tokens(tokens):
    '''POS-aware lemmatization of each token.'''
    tagged = pos_tag(tokens)
    return [lemmatizer.lemmatize(tok, _penn_to_wordnet_pos(tag)) for tok, tag in tagged]


demo_lemmas = lemmatize_tokens(demo_tokens_nostop)
print(demo_lemmas)


['cafe', 'resume', '18th', 'ed', 'naive', 'bayes', 'classification', 'system', 'see', 'email', 'approximately', 'versus', 'think', 'government', 'department', 'update', 'catalog', 'library']



### 3.9 Side-by-side comparison on the demo sentence


In [27]:

comparison = pd.DataFrame({
    "cleaned_token": demo_tokens_nostop,
    "stemmed": demo_stems,
    "lemmatized": demo_lemmas,
})
comparison


,cleaned_token,stemmed,lemmatized
0,cafe,cafe,cafe
1,resume,resum,resume
2,18th,18th,18th
3,ed,ed,ed
4,naive,naiv,naive
5,bayes,bay,bayes
6,classification,classif,classification
7,system,system,system
8,see,see,see
9,email,email,email



## 4. Assembling the full pipeline

All the individual steps above are combined into a single `CISIPreprocessor` class. Each stage
can be toggled on/off, and `transform()` returns the intermediate result of every stage so the
whole pipeline stays transparent and debuggable — useful both for reporting and for tuning the
pipeline on a new corpus.


In [28]:

class CISIPreprocessor:
    '''
    Configurable text preprocessing pipeline.

    Order of operations (matches the sections above):
      1. Expand contractions / slang & abbreviation normalization (text level, pre-tokenization)
      2. Case normalization (lowercasing)
      3. Noise removal at text level (URLs, emails, HTML)
      4. Diacritics removal
      5. Tokenization
      6. Noise removal at token level (punctuation, numbers, very short tokens)
      7. Abbreviation/slang normalization (token level)
      8. Stopword removal
      9. Stemming AND Lemmatization (parallel outputs, both computed from the same clean tokens)
    '''

    def __init__(
        self,
        use_stopwords=True,
        use_abbreviations=True,
        use_diacritics_removal=True,
        use_noise_removal=True,
        stopword_set=STOPWORDS,
        abbreviation_dict=ABBREVIATION_DICT,
    ):
        self.use_stopwords = use_stopwords
        self.use_abbreviations = use_abbreviations
        self.use_diacritics_removal = use_diacritics_removal
        self.use_noise_removal = use_noise_removal
        self.stopword_set = stopword_set
        self.abbreviation_dict = abbreviation_dict
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text: str) -> dict:
        '''Run the full pipeline on a single document; return every intermediate stage.'''
        stage_text = expand_contractions(text)
        stage_text = normalize_case(stage_text)
        if self.use_noise_removal:
            stage_text = remove_noise_text(stage_text)
        if self.use_diacritics_removal:
            stage_text = remove_diacritics(stage_text)

        tokens = tokenize(stage_text)

        if self.use_noise_removal:
            tokens = remove_noise_tokens(tokens)

        if self.use_abbreviations:
            tokens = normalize_abbreviations(tokens, self.abbreviation_dict)

        if self.use_stopwords:
            tokens_clean = remove_stopwords(tokens, self.stopword_set)
        else:
            tokens_clean = tokens

        stems = [self.stemmer.stem(tok) for tok in tokens_clean]
        lemmas = lemmatize_tokens(tokens_clean) if tokens_clean else []

        return {
            "clean_text": stage_text,
            "tokens": tokens_clean,
            "stems": stems,
            "lemmas": lemmas,
        }

    def transform_to_string(self, text: str, form: str = "lemmas") -> str:
        '''Convenience helper: run the pipeline and join the chosen token form back into a string.'''
        result = self.transform(text)
        return " ".join(result[form])


pipeline = CISIPreprocessor()
demo_result = pipeline.transform(demo_text)
demo_result


{'clean_text': 'cafe resume: the 18th ed. of the naive bayes classification system — see or e-mail it is approx. 1876 vs. 1971; do not you think the govt. dept. should update the catalog #libraries!!',
 'tokens': ['cafe',
  'resume',
  '18th',
  'ed',
  'naive',
  'bayes',
  'classification',
  'system',
  'see',
  'email',
  'approximately',
  'versus',
  'think',
  'government',
  'department',
  'update',
  'catalog',
  'libraries'],
 'stems': ['cafe',
  'resum',
  '18th',
  'ed',
  'naiv',
  'bay',
  'classif',
  'system',
  'see',
  'email',
  'approxim',
  'versu',
  'think',
  'govern',
  'depart',
  'updat',
  'catalog',
  'librari'],
 'lemmas': ['cafe',
  'resume',
  '18th',
  'ed',
  'naive',
  'bayes',
  'classification',
  'system',
  'see',
  'email',
  'approximately',
  'versus',
  'think',
  'government',
  'department',
  'update',
  'catalog',
  'library']}


## 5. Applying the pipeline to the full CISI corpus

We run the pipeline over every document's abstract (`.W` field) and store the tokenized,
stemmed, and lemmatized forms alongside the original text. This can take a little while for
1,460 documents but should complete in well under a minute.


In [29]:

def process_corpus(df: pd.DataFrame, pipeline: CISIPreprocessor, text_col: str = "text") -> pd.DataFrame:
    '''Apply the pipeline to every row of the corpus DataFrame and attach the results.'''
    tokens_col, stems_col, lemmas_col, clean_col = [], [], [], []
    for text in df[text_col].fillna(""):
        result = pipeline.transform(text)
        clean_col.append(result["clean_text"])
        tokens_col.append(result["tokens"])
        stems_col.append(result["stems"])
        lemmas_col.append(result["lemmas"])

    out = df.copy()
    out["clean_text"] = clean_col
    out["tokens"] = tokens_col
    out["stems"] = stems_col
    out["lemmas"] = lemmas_col
    return out


processed_df = process_corpus(corpus_df, pipeline, text_col="text")
processed_df[["doc_id", "tokens", "stems", "lemmas"]].head()


,doc_id,tokens,stems,lemmas
0,1,"[present, study, history, dewey, decimal, clas...","[present, studi, histori, dewey, decim, classi...","[present, study, history, dewey, decimal, clas..."
1,2,"[report, analysis, acts, use, technical, libra...","[report, analysi, act, use, technic, librari, ...","[report, analysis, act, use, technical, librar..."
2,3,"[relationships, organization, control, writing...","[relationship, organ, control, write, organ, c...","[relationship, organization, control, writings..."
3,4,"[establishment, nine, new, universities, provo...","[establish, nine, new, univers, provok, highli...","[establishment, nine, new, university, provoke..."
4,5,"[although, use, games, professional, education...","[although, use, game, profession, educ, becom,...","[although, use, game, professional, education,..."



### 5.1 Before / after example

Comparing one full document before and after preprocessing:


In [30]:

row = processed_df.iloc[0]
print("RAW TEXT:\n", row["text"], "\n")
print("TOKENS (post cleaning, stopword removal):\n", row["tokens"], "\n")
print("STEMMED:\n", row["stems"], "\n")
print("LEMMATIZED:\n", row["lemmas"])


RAW TEXT:
 The present study is a history of the DEWEY Decimal Classification.  The first edition of the DDC was published in 1876, the eighteenth edition in 1971, and future editions will continue to appear as needed.  In spite of the DDC's long and healthy life, however, its full story has never been told.  There have been biographies of Dewey that briefly describe his system, but this is the first attempt to provide a detailed history of the work that more than any other has spurred the growth of librarianship in this country and abroad. 

TOKENS (post cleaning, stopword removal):
 ['present', 'study', 'history', 'dewey', 'decimal', 'classification', 'first', 'edition', 'ddc', 'published', 'eighteenth', 'edition', 'future', 'editions', 'continue', 'appear', 'needed', 'spite', 'ddc', 'long', 'healthy', 'life', 'however', 'full', 'story', 'never', 'told', 'biographies', 'dewey', 'briefly', 'describe', 'system', 'first', 'attempt', 'provide', 'detailed', 'history', 'work', 'spurred', '

## 2. Fetch and Parse the CISI Corpus

The CISI collection contains information retrieval documents. We fetch the raw `.ALL` text file and extract document IDs, titles, authors, and text abstracts.

In [31]:
def load_cisi(path):
    """Looks out for the CISI.ALL file in a local path."""
    file_path = os.path.join(path, "CISI.ALL")

    # We look out for the file at the given path
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"No se encontró CISI.ALL en: {path}")

    # We read the document content
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_content = f.read()

    return raw_content

def parse_cisi_all(path):
    '''
    Parse a CISI.ALL file into a dict: {doc_id: {"title": str, "author": str, "text": str}}.

    The CISI.ALL format tags each field with a line starting with '.' followed by a single
    letter (I=id, T=title, A=author, W=body text, X=cross-references). Field content may span
    multiple lines until the next tag is encountered.
    '''

    raw_document = load_cisi(path)

    raw = raw_document.replace("\r\n", "\n")  # normalize Windows line endings
    tag_re = re.compile(r"^\.(I|T|A|W|X)\b\s?(.*)$")

    docs = {}
    doc_id = None
    field = None
    buf = {"T": "", "A": "", "W": ""}

    for line in raw.split("\n"):
        m = tag_re.match(line)
        if m:
            tag, rest = m.group(1), m.group(2)
            if tag == "I":
                if doc_id is not None:
                    docs[doc_id] = buf
                doc_id = int(rest.strip())
                buf = {"T": "", "A": "", "W": ""}
                field = None
            elif tag == "X":
                field = None  # ignore cross-reference block, not part of the document text
            else:
                field = tag
                if rest.strip():
                    buf[tag] += rest + " "
        else:
            if field in ("T", "A", "W"):
                buf[field] += line + " "

    if doc_id is not None:
        docs[doc_id] = buf

    records = [
        {
            "doc_id": did,
            "title": fields["T"].strip(),
            "author": fields["A"].strip(),
            "text": fields["W"].strip(),
        }
        for did, fields in sorted(docs.items())
    ]
    return pd.DataFrame(records)


In [32]:
# 1. We look out for the path where the notebook is saved
actual_folder = Path('.').resolve()

# 2. We go one folder up
upper_folder = actual_folder.parent

corpus_df = parse_cisi_all(upper_folder / 'data')
print(f"Loaded {len(corpus_df)} documents from {upper_folder / 'data'}")
corpus_df.head()

Loaded 1460 documents from C:\Users\TeamIT\Desktop\Jero\Maestria\NLP\Phase 02\data


,doc_id,title,author,text
0,1,18 Editions of the Dewey Decimal Classifications,"Comaromi, J.P.",The present study is a history of the DEWEY De...
1,2,Use Made of Technical Libraries,"Slater, M.",This report is an analysis of 6300 acts of use...
2,3,Two Kinds of Power An Essay on Bibliographic C...,"Wilson, P.",The relationships between the organization and...
3,4,Systems Analysis of a University Library; fin...,"Buckland, M.K.",The establishment of nine new universities in ...
4,5,A Library Management Game: a report on a resea...,"Brophy, P.",Although the use of games in professional educ...
